In [ ]:
# cell 01
import sagemaker
bucket=sagemaker.Session().default_bucket()
prefix = 'sagemaker'
 
# Define IAM role
import boto3
import re
from sagemaker import get_execution_role

role = get_execution_role()
role

In [ ]:
# cell 02
import os
import boto3
import pandas as pd
from sklearn.model_selection import train_test_split

s3 = boto3.client("s3")
region = boto3.session.Session().region_name

# 列名を付与（元CSVはヘッダー無し）
cols = ["Sex","Length","Diameter","Height","Whole","Shucked","Viscera","Shell","Rings"]
df = pd.read_csv("abalone.csv", header=None, names=cols)

# Sex -> 数値化
sex_map = {"I":0, "F":1, "M":2}
df["SexCode"] = df["Sex"].map(sex_map).astype(int)

# XGBoost入力用: ラベル先頭 + 特徴量（8列）
features = ["Length","Diameter","Height","Whole","Shucked","Viscera","Shell","SexCode"]
proc = pd.concat([df["Rings"], df[features]], axis=1)

# 学習/検証に分割
train_df, val_df = train_test_split(proc, test_size=0.2, random_state=42)

# ヘッダー無しCSVで保存
os.makedirs("data", exist_ok=True)
train_path = "data/train.csv"
val_path   = "data/validation.csv"
train_df.to_csv(train_path, header=False, index=False)
val_df.to_csv(val_path,   header=False, index=False)

# S3 へアップロード（prefix は Cell 01 で 'sagemaker/data' などを指定済み）
train_key = f"{prefix}/train/train.csv"
val_key   = f"{prefix}/validation/validation.csv"
s3.upload_file(train_path, bucket, train_key)
s3.upload_file(val_path,   bucket, val_key)

train_s3_uri = f"s3://{bucket}/{train_key}"
val_s3_uri   = f"s3://{bucket}/{val_key}"
print("Train:", train_s3_uri)
print("Validation:", val_s3_uri)

In [ ]:
# cell 03
container = sagemaker.image_uris.retrieve(region=boto3.Session().region_name, framework='xgboost', version='latest')
container

In [ ]:
# cell 04
import sagemaker
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator

sess = sagemaker.Session()

xgb = sagemaker.estimator.Estimator(container,
                                    role, 
                                    instance_count=1, 
                                    instance_type='ml.m5.xlarge',
                                    output_path='s3://{}/{}/output'.format(bucket, prefix),
                                    sagemaker_session=sess)

# 回帰（Rings を予測）設定
xgb.set_hyperparameters(
    objective="reg:linear",
    eval_metric="rmse",
    num_round=200,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
)



# 入力チャンネル（CSV, ラベルが先頭）
train_input = TrainingInput(train_s3_uri, content_type="text/csv")
val_input   = TrainingInput(val_s3_uri,   content_type="text/csv")


xgb.fit({"train": train_input, "validation": val_input}, logs=True)

print("Model artifacts:", xgb.model_data)  # s3://.../model.tar.gz

In [ ]:
# # モデルだけ作成
# from sagemaker.model import Model

# # すでに fit() 終了後なので、xgb.model_data に model.tar.gz が入っている
# print("Training completed. Model artifacts at:", xgb.model_data)

# # XGBoost のデフォルト推論イメージを使用してモデル作成
# model = Model(
#     # name='model_name',          # モデル名を指定する場合
#     image_uri=container,      # cell 03 の XGBoost イメージ
#     model_data=xgb.model_data,
#     role=role,
#     sagemaker_session=sess
# )

# model_name = model.create()
# print("Created model name:", model_name)

In [ ]:
# モデル、エンドポイント設定、エンドポイントが作成される
predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.xlarge'
)

In [ ]:
from sagemaker.serializers import CSVSerializer

predictor.serializer = CSVSerializer()      # ← これで ContentType='text/csv' が自動セット
result = predictor.predict([0.5,0.44,0.15,0.08,0.2,0.07,0.09,0.05])
result

# 自分のtrain.py, inference.pyを使用する場合

In [ ]:
# import sagemaker
# from sagemaker.inputs import TrainingInput
# from sagemaker.estimator import Estimator

# sess = sagemaker.Session()

# # ★ train.py / inference.py を内包する sourcedir.tar.gz の S3 パス
# submit_dir = f"s3://{bucket}/{prefix}/code/sourcedir.tar.gz"

# # Estimator（Script Mode を強制 ON）
# xgb = Estimator(
#     image_uri=container,
#     role=role,
#     instance_count=1,
#     instance_type="ml.m5.xlarge",
#     output_path=f"s3://{bucket}/{prefix}/output",
#     sagemaker_session=sess,

#     hyperparameters={
#         "sagemaker_program": "train.py",     # 実行するスクリプト
#         "sagemaker_submit_directory": submit_dir,

#         # train.py に渡すパラメータ
#         "objective": "reg:squarederror",
#         "eval_metric": "rmse",
#         "num_round": 200,
#         "max_depth": 5,
#         "eta": 0.2,
#         "subsample": 0.8,
#         "colsample_bytree": 0.8
#     }
# )

# # 入力チャネル
# train_input = TrainingInput(train_s3_uri, content_type="text/csv")
# val_input   = TrainingInput(val_s3_uri,   content_type="text/csv")

# # 学習開始
# xgb.fit({"train": train_input, "validation": val_input}, logs=True)

# print("Model artifacts:", xgb.model_data)

In [ ]:
# from sagemaker.model import Model

# # ② inference.py を entry_point に指定して推論モデルを作成
# inference_model = Model(
#     image_uri=container,
#     model_data=xgb.model_data,
#     role=role,

#     # ★ submit_dir をそのまま使う（inference_submit_dir ではない）
#     source_dir=submit_dir,
#     entry_point="inference.py",

#     sagemaker_session=sess
# )

# # ③ Model 登録
# model_name = inference_model.create()
# print("Created model name:", model_name)